In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

warnings.filterwarnings("ignore")


df = pd.read_csv("/workspaces/f1-project/02-data/master_qualifying_data_enriched.csv")
df["Quali_Position"] = df["Quali_Position"].astype(int)

print("✅ Dataset ENRIQUECIDO carregado com colunas de Pontos e Vitórias.")

In [ ]:
# METADATA

print("Aplicando Metadados de Pista (Physics-Awareness)...")

# Dicionário de Características (0-10 ou Categorias)
# Downforce: 1 (Baixo/Rápido) a 5 (Alto/Lento)
# Overtaking: 1 (Difícil/Monaco) a 5 (Fácil/Vegas)
TRACK_STATS = {
    "Bahrain Grand Prix": {"Downforce": 3, "Overtaking": 4, "Type": "Permanent"},
    "Saudi Arabian Grand Prix": {"Downforce": 2, "Overtaking": 4, "Type": "Street"},
    "Australian Grand Prix": {"Downforce": 4, "Overtaking": 3, "Type": "Street_Hybrid"},
    "Azerbaijan Grand Prix": {"Downforce": 2, "Overtaking": 5, "Type": "Street"},
    "Miami Grand Prix": {"Downforce": 3, "Overtaking": 3, "Type": "Street"},
    "Monaco Grand Prix": {"Downforce": 5, "Overtaking": 1, "Type": "Street"},
    "Spanish Grand Prix": {"Downforce": 4, "Overtaking": 3, "Type": "Permanent"},
    "Canadian Grand Prix": {"Downforce": 2, "Overtaking": 4, "Type": "Permanent"},
    "Austrian Grand Prix": {"Downforce": 3, "Overtaking": 4, "Type": "Permanent"},
    "British Grand Prix": {"Downforce": 4, "Overtaking": 4, "Type": "Permanent"},
    "Hungarian Grand Prix": {"Downforce": 5, "Overtaking": 2, "Type": "Permanent"},
    "Belgian Grand Prix": {"Downforce": 2, "Overtaking": 4, "Type": "Permanent"},
    "Dutch Grand Prix": {"Downforce": 5, "Overtaking": 2, "Type": "Permanent"},
    "Italian Grand Prix": {
        "Downforce": 1,
        "Overtaking": 4,
        "Type": "Permanent",
    },  # Monza
    "Singapore Grand Prix": {"Downforce": 5, "Overtaking": 2, "Type": "Street"},
    "Japanese Grand Prix": {"Downforce": 5, "Overtaking": 3, "Type": "Permanent"},
    "Qatar Grand Prix": {"Downforce": 4, "Overtaking": 3, "Type": "Permanent"},
    "United States Grand Prix": {"Downforce": 4, "Overtaking": 4, "Type": "Permanent"},
    "Mexico City Grand Prix": {"Downforce": 5, "Overtaking": 3, "Type": "Permanent"},
    "São Paulo Grand Prix": {"Downforce": 3, "Overtaking": 4, "Type": "Permanent"},
    "Las Vegas Grand Prix": {"Downforce": 1, "Overtaking": 5, "Type": "Street"},
    "Abu Dhabi Grand Prix": {"Downforce": 3, "Overtaking": 3, "Type": "Permanent"},
    "Chinese Grand Prix": {"Downforce": 4, "Overtaking": 4, "Type": "Permanent"},
    "Emilia Romagna Grand Prix": {"Downforce": 4, "Overtaking": 2, "Type": "Permanent"},
}


# Aplicar ao DataFrame
def apply_track_stats(row):
    event = row["Event"]
    stats = TRACK_STATS.get(
        event, {"Downforce": 3, "Overtaking": 3, "Type": "Permanent"}
    )  # Default
    return pd.Series([stats["Downforce"], stats["Overtaking"], stats["Type"]])


df[["Track_Downforce", "Track_Overtaking", "Track_Type"]] = df.apply(
    apply_track_stats, axis=1
)

# Converter 'Track_Type' para números (Dummies) agora ou deixar para o pd.get_dummies depois
print("✅ Metadados de pista aplicados.")

In [ ]:
# 1. Gaps relativos
for session in ["FP1_Time", "FP2_Time", "FP3_Time"]:
    p1_time = df.groupby(["Year", "Event"])[session].transform("min")
    df[f"Gap_{session}"] = df[session] - p1_time
    # Percentil em cada sessão
    df[f"Rank_{session}"] = df.groupby(["Year", "Event"])[session].rank()
    df[f"Percentile_{session}"] = df.groupby(["Year", "Event"])[session].rank(pct=True)

# 2. Agregações dos gaps
df["Avg_Gap"] = df[["Gap_FP1_Time", "Gap_FP2_Time", "Gap_FP3_Time"]].mean(axis=1)
df["Min_Gap"] = df[["Gap_FP1_Time", "Gap_FP2_Time", "Gap_FP3_Time"]].min(axis=1)
df["Max_Gap"] = df[["Gap_FP1_Time", "Gap_FP2_Time", "Gap_FP3_Time"]].max(axis=1)
df["Std_Gap"] = df[["Gap_FP1_Time", "Gap_FP2_Time", "Gap_FP3_Time"]].std(axis=1)

# 3. Agregações dos ranks
df["Avg_Rank"] = df[["Rank_FP1_Time", "Rank_FP2_Time", "Rank_FP3_Time"]].mean(axis=1)
df["Best_Rank"] = df[["Rank_FP1_Time", "Rank_FP2_Time", "Rank_FP3_Time"]].min(axis=1)
df["Worst_Rank"] = df[["Rank_FP1_Time", "Rank_FP2_Time", "Rank_FP3_Time"]].max(axis=1)

# 4. Agregações dos percentis
df["Avg_Percentile"] = df[
    ["Percentile_FP1_Time", "Percentile_FP2_Time", "Percentile_FP3_Time"]
].mean(axis=1)

# 5. Consistência
df["Gap_Range"] = df["Max_Gap"] - df["Min_Gap"]
df["Rank_Range"] = df["Worst_Rank"] - df["Best_Rank"]
df["Gap_CV"] = df["Std_Gap"] / (df["Avg_Gap"].abs() + 0.001)  # Coeficiente de variação

# 6. Tendência de evolução
df["Trend_Gap"] = df["Gap_FP3_Time"] - df["Gap_FP1_Time"]  # Negativo = melhorou
df["Trend_Rank"] = df["Rank_FP3_Time"] - df["Rank_FP1_Time"]

# 7. Estatísticas de tempos absolutos
df["Avg_FP_Time"] = df[["FP1_Time", "FP2_Time", "FP3_Time"]].mean(axis=1)
df["Min_FP_Time"] = df[["FP1_Time", "FP2_Time", "FP3_Time"]].min(axis=1)

# 8. Ordenar por tempo
df = df.sort_values(["Year", "Event", "Driver"])

# 9. Histórico do piloto (rolling mais sofisticado)
df = df.sort_values(["Driver", "Year", "Event"])

# Últimas N corridas (médias móveis)
for window in [3, 5, 10]:
    df[f"Driver_Last{window}_Avg"] = df.groupby("Driver")["Quali_Position"].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
    )
    df[f"Driver_Last{window}_Best"] = df.groupby("Driver")["Quali_Position"].transform(
        lambda x: x.rolling(window=window, min_periods=1).min().shift(1)
    )

# Tendência recente (últimas 5 vs últimas 10)
df["Driver_Recent_Trend"] = df["Driver_Last5_Avg"] - df["Driver_Last10_Avg"]

# Desvio padrão (consistência)
df["Driver_Std"] = df.groupby("Driver")["Quali_Position"].transform(
    lambda x: x.expanding().std().shift(1)
)

# 10. Histórico da equipe
df = df.sort_values(["Team", "Year", "Event"])

for window in [3, 5]:
    df[f"Team_Last{window}_Avg"] = df.groupby("Team")["Quali_Position"].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
    )

# 11. Performance no circuito específico
df["Driver_Track_Avg"] = df.groupby(["Driver", "Event"])["Quali_Position"].transform(
    lambda x: x.expanding().mean().shift(1)
)
df["Driver_Track_Best"] = df.groupby(["Driver", "Event"])["Quali_Position"].transform(
    lambda x: x.expanding().min().shift(1)
)

df["Team_Track_Avg"] = df.groupby(["Team", "Event"])["Quali_Position"].transform(
    lambda x: x.expanding().mean().shift(1)
)

# 12. Interações importantes
df["Gap_x_DriverAvg"] = df["Avg_Gap"] * df["Driver_Last5_Avg"]
df["Rank_x_TeamAvg"] = df["Avg_Rank"] * df["Team_Last5_Avg"]

In [ ]:
# ==============================================================================
# ❄️ COLD START HANDICAP ENGINE (2026 PREP - Issue #3)
# ==============================================================================
def apply_cold_start_handicap(df):
    """
    Imputa valores de 'Fundo de Grid' para equipes novas (Cadillac)
    que não possuem histórico, evitando que o modelo assuma a média global.
    """
    # 1. Definição da Penalidade
    rookie_teams = ["Cadillac"]
    HANDICAP_POS = 19.0  # Assumimos que largam no fim do grid

    print(f"\n❄️ Verificando necessidade de Handicap para: {rookie_teams}")

    # Verifica se existem carros dessas equipes no dataset
    mask_rookie = df["Team"].isin(rookie_teams)

    if mask_rookie.sum() == 0:
        print(
            "   -> Nenhuma equipe nova encontrada neste dataset (OK para dados históricos)."
        )
        return df

    print(f"   -> Encontradas {mask_rookie.sum()} entradas de equipes novas.")

    # 2. Identificar colunas históricas criadas acima
    cols_to_fix = [
        "Driver_Last10_Avg",
        "Driver_Last3_Avg",
        "Driver_Last5_Avg",
        "Team_Last5_Avg",
        "Team_Last3_Avg",
        "Driver_Last10_Best",
        "Driver_Last5_Best",
        "Driver_Recent_Trend",  # Importante zerar ou ajustar a tendência
    ]

    # 3. Aplicação do Handicap
    for col in cols_to_fix:
        if col in df.columns:
            # Conta quantos NaNs existem nessas equipes
            nans_before = df.loc[mask_rookie, col].isna().sum()

            if nans_before > 0:
                # Preenche NaN com 19.0 (Fundo do Grid)
                df.loc[mask_rookie & df[col].isna(), col] = HANDICAP_POS
                print(
                    f"      -> Imputado P{HANDICAP_POS} em '{col}' para {nans_before} linhas."
                )

    # Correção específica para Tendência (Trend deve ser neutra ou negativa, não 19)
    if "Driver_Recent_Trend" in df.columns:
        df.loc[
            mask_rookie & df["Driver_Recent_Trend"].isna(), "Driver_Recent_Trend"
        ] = 0.0

    return df


# 🔥 APLICA A LÓGICA IMEDIATAMENTE
df = apply_cold_start_handicap(df)

In [ ]:
print("Criando dummies para Team e Driver...")
df_model = pd.get_dummies(df, columns=["Team", "Driver"], drop_first=True)

# Usando a nossa lista de "Top 20 Features" (do Modelo Enxuto)
top_features = [
    "Rank_x_TeamAvg",
    "Avg_Rank",
    "Avg_Percentile",
    "Gap_x_DriverAvg",
    "Driver_Last10_Avg",
    "Best_Rank",
    "Rank_FP3_Time",
    "Percentile_FP3_Time",
    "Min_Gap",
    "Avg_Gap",
    "Driver_Last3_Avg",
    "Driver_Last5_Avg",
    "Team_Last5_Avg",
    "Worst_Rank",
    "Driver_Last10_Best",
    "Driver_Last5_Best",
    "Gap_FP3_Time",
    "Rank_FP2_Time",
    "Percentile_FP2_Time",
    "Gap_FP2_Time",
    # --- NOVAS FEATURES ---
    "Track_Downforce",  # O carro gosta dessa pista?
    "Track_Overtaking",  # É fácil passar?
    "Driver_Points",  # O piloto é bom de campeonato?
    "Driver_Wins",  # O piloto é vencedor?
    "Championship_Pos",  # P1 do campeonato vs P20
]

# Garantir que as colunas existem no df_model
features_validas = [col for col in top_features if col in df_model.columns]

print(f"Selecionadas {len(features_validas)} features para o modelo.")

X = df_model[features_validas]
y = df_model["Quali_Position"]

In [ ]:
# ========================================
# TREINAMENTO COM XGBOOST (MOTOR V2)
# ========================================
from xgboost import XGBRegressor

print("Iniciando treinamento com XGBoost...")

# 1. Split (Mantemos igual)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# 2. Definir o Modelo Base XGBoost
xgb_model = XGBRegressor(
    objective="reg:absoluteerror",  # Otimizar para Erro Absoluto (MAE) - perfeito para posições
    n_jobs=-1,
    random_state=42,
)

# 3. Definir a Grade de Hiperparâmetros (O Segredo do XGBoost)
param_grid = {
    "n_estimators": [100, 300, 500],  # Quantas árvores criar
    "learning_rate": [
        0.01,
        0.05,
        0.1,
    ],  # A velocidade de aprendizado (menor é mais preciso, mas lento)
    "max_depth": [3, 5, 7],  # Profundidade da árvore (evita overfitting)
    "subsample": [0.7, 0.9],  # Usar apenas uma parte dos dados por árvore (evita vício)
    "colsample_bytree": [0.7, 0.9],  # Usar apenas algumas features por árvore
}

# 4. Grid Search
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",  # Focar no MAE
    verbose=1,
    n_jobs=-1,
)

print("Otimizando hiperparâmetros (isso pode demorar um pouco)...")
grid_search.fit(X_train, y_train)

# 5. Melhor Modelo
best_model = grid_search.best_estimator_

print("\n✅ Treinamento concluído!")
print(f"Melhores parâmetros: {grid_search.best_params_}")

In [ ]:
# ========================================
# AVALIAÇÃO
# ========================================
pred_train = best_model.predict(X_train)
pred_test = best_model.predict(X_test)

# Arredondar (pois posições são inteiros)
pred_test_rounded = np.round(np.clip(pred_test, 1, 20))
pred_train_rounded = np.round(np.clip(pred_train, 1, 20))

print("\n--- TREINO (XGBoost) ---")
print(f"R²: {r2_score(y_train, pred_train):.3f}")
print(f"MAE: {mean_absolute_error(y_train, pred_train_rounded):.3f} posições")

print("\n--- TESTE (XGBoost) ---")
print(f"R²: {r2_score(y_test, pred_test):.3f}")
print(f"MAE: {mean_absolute_error(y_test, pred_test_rounded):.3f} posições")

In [ ]:
# ========================================
# 6. SALVAR O MODELO (IMPORTANTE)
# ========================================
import joblib

# Salvar com um nome novo para sabermos que é o XGBoost
caminho_modelo = "/workspaces/f1-project/03-models/xgb_model_quali_v1.pkl"
caminho_features = "/workspaces/f1-project/03-models/xgb_features_quali_v1.pkl"

joblib.dump(best_model, caminho_modelo)
joblib.dump(features_validas, caminho_features)

print(f"\n✅ Modelo XGBoost salvo em: {caminho_modelo}")
print(f"✅ Lista de Features salva em: {caminho_features}")

In [ ]:
# ========================================
# 7. Model Explainability (SHAP)
# ========================================
import matplotlib.pyplot as plt
import shap

# Inicializa visualização JS (necessário para alguns gráficos interativos)
shap.initjs()

print("🤖 Calculando valores SHAP... (Isso pode levar alguns segundos)")

# 1. Cria o Explainer (O componente que entende o modelo)
# O SHAP tenta adivinhar o melhor explainer (TreeExplainer para árvores, Linear para regressão, etc)
explainer = shap.Explainer(best_model)

# 2. Calcula os valores SHAP para o nosso conjunto de teste
# Isso explica "Por que o modelo previu X para esses dados de teste?"
shap_values = explainer(X_test)

# 3. Gráfico Global (Beeswarm Plot) - A "Jóia da Coroa" do SHAP
# Mostra quais features importam mais e como elas impactam (positivo/negativo)
plt.title("Impacto das Features na Classificação Final", fontsize=14)
shap.plots.beeswarm(shap_values, max_display=15)  # Mostra as top 15 features

In [ ]:
# 8. Salvar o Explainer

import joblib

# Salva o explainer na pasta de modelos
joblib.dump(explainer, "/workspaces/f1-project/03-models/shap_explainer_v1.pkl")
print("Explainer salvo com sucesso!")